a)Selecione todos os clientes paulistanos

Pedido com o Maior e Menor frete ?

In [0]:
%sql
with fretes as (
select 
  ROW_NUMBER() OVER (ORDER BY freight_value desc ) as maior_frete, 
  ROW_NUMBER() OVER (ORDER BY freight_value ) as menor_frete,
  order_id, order_item_id, product_id, seller_id, price, freight_value
from tmw_olist.order_items
)
select 
  max(case 
    when maior_frete = 1 then order_id  end) as Pedido_maior_frete ,
  max(case     
    when menor_frete = 1 then order_id  end ) as Pedido_menor_frete 
from fretes 


Qual Cliente tem mais pedidos

In [0]:
%sql
select 
  b.customer_unique_id,
  count(a.order_id) as qt_pedidos 
 
      from 
  workspace.tmw_olist.orders  a 
  left join     workspace.tmw_olist.customers b 
    on a.customer_id = b.customer_id
group by all 
order by qt_pedidos desc 
limit 1 





Qual vendedor tem mais itens vendidos? 
     E o com menos?	

In [0]:
%sql
with itens as (
  select 
      seller_id,
      count(order_item_id) as qt_itens 
  from 
      tmw_olist.order_items
 group by seller_id
),
Maior_Menor as (
  select *,
         row_number() over (order by qt_itens desc) as Vendedor_Maior, 
         row_number() over (order by qt_itens asc) as Vendedor_Menor
    from itens
)
select 
  max(case when Vendedor_Maior = 1 then seller_id end) as Vendedor_ID_maior,
  max(case when Vendedor_Menor = 1 then seller_id end) as Vendedor_ID_menor
from Maior_Menor



Qual do ano dia tivemos mais pedidos?

In [0]:
%sql
select 
    cast(date_format(order_purchase_timestamp, "yyyy-MM-dd") as date )as Dt_Pedido, 
    count(order_id ) as Qt_Pedidos
     from
         workspace.tmw_olist.orders 
group by all 
order by Qt_Pedidos desc 
limit 1


2) A) Quantos vendedores são do estado de São Paulo?

In [0]:
%sql
select 
  seller_state ,
  count(seller_id) as qt from workspace.tmw_olist.sellers
where 
  seller_state = 'SP'
group by all 


B) Quantos vendedores são de Presidente Prudente?

In [0]:
%sql
select 
  seller_city ,
  count(seller_id) as qt from workspace.tmw_olist.sellers
where 
  seller_state = 'SP' and seller_city like '%presidente prudente%'
group by all 

C) Quantos clientes são do estado do Rio de Janeiro?

In [0]:
%sql
select 
  customer_state ,
  count(customer_id) as qt from workspace.tmw_olist.customers
where 
  customer_state = 'RJ'
group by all 

D) Quantos produtos são de construção? Qualquer tipo de construção. 

In [0]:
%sql
select 
  product_category_name,
  count(product_id) as qt 
from 
  workspace.tmw_olist.products
 where 
   product_category_name like '%constru%'
group by all 



E) Qual o valor médio de um pedido? E do frete?

In [0]:
%sql
select 
  mean(price) as valor_medio, 
  mean(freight_value) as frete_medio
from 
  workspace.tmw_olist.order_items

group by all 



F) Em média os pedidos são de quantas parcelas de cartão? E o valor médio por parcela?

In [0]:
%sql
select 
  mean(payment_installments) as Qt_Parc_Media,
  mean(case 
    when payment_installments = 0 then payment_value else (payment_value/payment_installments) end) as Vl_Medio_Parcela
from 
  workspace.tmw_olist.order_payments
group by all 


8 12.41625
1 24.39
1 65.71

G) Quanto tempo em média demora para um pedido chegar depois de aprovado?


In [0]:
%sql
select 
   mean(date_diff(order_estimated_delivery_date,order_approved_at)) as Tempo_Medio_Dias
   from workspace.tmw_olist.orders
where order_approved_at is not null 
group by all



3. A)Qual estado tem mais vendedores?

In [0]:
%sql
select 
  seller_state,
  count(*) as qt 
   from workspace.tmw_olist.sellers
  group by all 
  order by qt desc 
limit 1

  

3. B) Qual cidade tem mais clientes?

In [0]:
%sql
select 
  customer_state,
  count(*) as qt 
   from workspace.tmw_olist.customers
  group by all 
  order by qt desc 
limit 1

3 C) Qual categoria tem mais itens?

In [0]:
%sql
select 
  product_category_name,
  count(product_id) as qt
 from 
    workspace.tmw_olist.products
  group by all 
  order by qt desc 
  limit 1
  

3 D) Qual categoria tem maior peso médio de produto?


In [0]:
%sql
select 
    product_category_name,
  mean(product_weight_g) as peso_medio
 from 
    workspace.tmw_olist.products
  group by all 
  order by peso_medio desc
limit 1


3. E) Qual a série histórica de pedidos por dia? E receita? E por Mês


In [0]:
%sql
select
   cast(date_format(a.order_purchase_timestamp, 'yyyy-MM-dd')  as date )as DataVenda,
   count(a.order_id) as qt,
   sum(b.payment_value) as receita 
from 
  workspace.tmw_olist.orders a

  left join tmw_olist.order_payments b  
    on a.order_id = b.order_id
group by all
order by DataVenda



In [0]:

%sql
select
   cast(date_format(date_trunc("MONTH",order_purchase_timestamp), 'yyyy-MM-dd') as date )as AnoMes_Venda,
   count(a.order_id) as qt,
   sum(b.payment_value) as receita 
from 
  workspace.tmw_olist.orders a

  left join tmw_olist.order_payments b  
    on a.order_id = b.order_id
group by all
order by AnoMes_Venda




FIM FIM FIM 


Lista de pedidos com mais de um item.

In [0]:
%sql
select order_id,count(product_id) as quantidade_itens from tmw_olist.order_items
group by all 
having quantidade_itens > 1

Lista de pedidos que o frete é mais caro que o item.

In [0]:
%sql
select * from tmw_olist.order_items
where freight_value > price 

Lista de pedidos que ainda não foram enviados.

In [0]:
%sql
select * from tmw_olist.orders
where order_status not in ("delivered","shipped")

Lista de pedidos que foram entregues com atraso.

In [0]:
%sql
select 
*
    from tmw_olist.orders
 where  order_estimated_delivery_date < order_delivered_customer_date

Lista de pedidos que foram entregues com 2 dias de antecedência.

In [0]:
%sql
select 
*,
DATEDIFF(day, order_delivered_customer_date, order_estimated_delivery_date) as diferenca
    from tmw_olist.orders
    group by all 
    
 having diferenca = 2

Lista de pedidos feitos em dezembro de 2017 e entregues com atraso

In [0]:
%sql
select 
*
    from tmw_olist.orders
    where month(order_purchase_timestamp) = 12 and year(order_purchase_timestamp) = 2017 and
    order_estimated_delivery_date < order_delivered_customer_date
    group by all 

Lista de pedidos com avaliação maior ou igual que 4

In [0]:
%sql
select 
*
    from workspace.tmw_olist.order_reviews
where review_score >= 4


Lista de pedidos com 2 ou mais parcelas menores que R$20,00

In [0]:
%sql
select 
  *
   from workspace.tmw_olist.order_payments 
   where payment_installments >= 2 and payment_value > 20

Selecione todos os pedidos e marque se houve atraso ou não

In [0]:
%sql
select 
*,
order_estimated_delivery_date < order_delivered_customer_date as atraso 
    from tmw_olist.orders

Selecione os pedidos/itens e defina os grupos em uma nova coluna:

In [0]:
%sql
select 
* ,
    case 
        when (freight_value/price) < 0.10 then "10%"
         when (freight_value/price) >= 0.10 and (freight_value/price)  < 0.25 then "10% a 25%"
         when (freight_value/price) >= 0.25 and (freight_value/price)  < 0.50 then "25% a 50%"
         when (freight_value/price) >= 0.50 then "Maior que 50%"
    end as Peso_Frete        
    
    from workspace.tmw_olist.order_items

Selecione a tabela silver.olist.produto :

In [0]:
%sql
select 
*,

    case 
    
        when product_category_name in ('alimentos' ,'alimentos_bebidas') then 'alimentos'
        when product_category_name in ('artes' ,'artes_e_artesanato') then 'artes'
        when  product_category_name in ('construcao_ferramentas_seguranca',
        'casa_construcao',
        'construcao_ferramentas_construcao',
        'construcao_ferramentas_ferramentas',
        'construcao_ferramentas_iluminacao',
        'construcao_ferramentas_jardim') then 'contrução'
    end as descNovaCategoria,

    case 
        when product_weight_g < 2000 then 'leve'
        when product_weight_g >=  2000  and product_weight_g < 5000 then 'medio'
        when product_weight_g >=  5000  and product_weight_g < 10000 then 'pesado'
        when product_weight_g >=  10000  then 'muito pesado'
     end as descPeso
     

       
        from tmw_olist.products
  group by all 